# 🧬 Self-Replicating Agent — Kaggle GPU Edition

**GPU:** T4 (16 GB VRAM)  
**Model:** `qwen2.5-coder:14b` (~9 GB — fits easily on T4, 4× better than 7B)  
**Speed:** ~30–40 tok/s GPU vs ~2 tok/s CPU

### Before running:
1. `Settings → Accelerator → GPU T4 x1`  ✅
2. `Settings → Internet → On`  ✅
3. Add your GitHub token as a Kaggle Secret named `GITHUB_TOKEN` (if repo is private)

### Optional — add API keys for cloud fallback:
Add these as Kaggle Secrets (Settings → Secrets):
- `GROQ_API_KEY`
- `CEREBRAS_API_KEY`

In [ ]:
# ── Cell 1: Verify GPU ─────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — enable GPU in Settings!')
import os
print('CUDA:', os.environ.get('CUDA_VISIBLE_DEVICES', 'not set'))

In [ ]:
# ── Cell 2: Install Ollama ─────────────────────────────────────────────
import subprocess, os

print('Installing Ollama...')
r = subprocess.run(
    'curl -fsSL https://ollama.com/install.sh | sh',
    shell=True, capture_output=True, text=True
)
if r.returncode != 0:
    print('STDERR:', r.stderr[-500:])
else:
    print('✅ Ollama installed')

# Verify
v = subprocess.run(['ollama', '--version'], capture_output=True, text=True)
print(v.stdout.strip())

In [ ]:
# ── Cell 3: Start Ollama server with GPU ──────────────────────────────
import subprocess, time, urllib.request, json

env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'
env['OLLAMA_KEEP_ALIVE'] = '24h'  # keep model loaded between calls
env['OLLAMA_NUM_PARALLEL'] = '1'  # single user, no need for parallel

server = subprocess.Popen(
    ['ollama', 'serve'],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
print(f'Ollama server PID: {server.pid}')
time.sleep(4)

# Verify server is up
try:
    with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=5) as r:
        print('✅ Ollama server running')
except Exception as e:
    print(f'❌ Server not responding: {e}')

In [ ]:
# ── Cell 4: Pull qwen2.5-coder:14b (fits on T4 16GB) ─────────────────
# ~9 GB download — takes 3-5 min on Kaggle's fast connection
import subprocess, sys

MODEL = 'qwen2.5-coder:14b'
print(f'Pulling {MODEL} (~9 GB)...')

r = subprocess.run(['ollama', 'pull', MODEL],
                   capture_output=True, text=True, timeout=600)
if r.returncode == 0:
    print(f'✅ {MODEL} ready')
else:
    print('STDERR:', r.stderr[-300:])

# Verify GPU is being used
import urllib.request, json
with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=5) as resp:
    models = json.load(resp).get('models', [])
    for m in models:
        gb = m.get('size', 0) / 1e9
        print(f"  {m['name']:40s} {gb:.1f} GB")

In [ ]:
# ── Cell 5: Clone the evolution code ─────────────────────────────────
import subprocess, os
from kaggle_secrets import UserSecretsClient

REPO = 'https://github.com/balaji33k/SelfReplicatingAgent.git'
BRANCH = 'fresh-main'
DEST = '/kaggle/working/SelfReplicatingAgent'

# Try to get GitHub token for private repo (add as Kaggle Secret if needed)
try:
    secrets = UserSecretsClient()
    token = secrets.get_secret('GITHUB_TOKEN')
    REPO = REPO.replace('https://', f'https://{token}@')
    print('Using GitHub token for private repo')
except Exception:
    print('No GITHUB_TOKEN — assuming public repo')

if os.path.exists(DEST):
    subprocess.run(['git', '-C', DEST, 'pull'], check=True)
    print('✅ Repo updated')
else:
    r = subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO, DEST],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f'✅ Cloned to {DEST}')
    else:
        print('❌ Clone failed:', r.stderr)

# List what we have
subprocess.run(['ls', '-la', f'{DEST}/generations/gen_1/'], check=True)

In [ ]:
# ── Cell 6: Install Python dependencies ──────────────────────────────
import subprocess, sys

DEST = '/kaggle/working/SelfReplicatingAgent'

pkgs = [
    'langchain-groq',
    'langchain-core',
    'langgraph',
    'langchain-community',
    'langchain-google-genai',
    'cerebras-cloud-sdk',
    'openai',          # for SambaNova + OpenRouter (OpenAI-compatible)
]

print('Installing dependencies...')
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + pkgs,
    capture_output=True, text=True
)
if r.returncode == 0:
    print('✅ Dependencies installed')
else:
    print('STDOUT:', r.stdout[-300:])
    print('STDERR:', r.stderr[-300:])

In [ ]:
# ── Cell 7: Configure environment ────────────────────────────────────
import os, json
from kaggle_secrets import UserSecretsClient

DEST = '/kaggle/working/SelfReplicatingAgent'

# Point to local Ollama with 14b model
os.environ['OLLAMA_BASE_URL'] = 'http://localhost:11434'
os.environ['OLLAMA_MODEL'] = 'qwen2.5-coder:14b'

# Load optional cloud API keys from Kaggle Secrets (for fallback chain)
secrets = UserSecretsClient()
for key in ['GROQ_API_KEY', 'CEREBRAS_API_KEY', 'SAMBANOVA_API_KEY',
            'OPENROUTER_API_KEY', 'GoogleAPIKey']:
    try:
        val = secrets.get_secret(key)
        if val:
            os.environ[key] = val
            print(f'✅ {key} loaded from secrets')
    except Exception:
        pass  # secret not set — that's fine

# Write user_config.json so llm_client picks up the right model
os.makedirs(f'{DEST}/data', exist_ok=True)
with open(f'{DEST}/data/user_config.json', 'w') as f:
    json.dump({'model': 'qwen2.5-coder:14b'}, f)

print(f'\nActive config:')
print(f'  OLLAMA_BASE_URL = {os.environ["OLLAMA_BASE_URL"]}')
print(f'  OLLAMA_MODEL    = {os.environ["OLLAMA_MODEL"]}')

In [ ]:
# ── Cell 8: Quick LLM smoke test (before full run) ───────────────────
import urllib.request, json, time

MODEL = 'qwen2.5-coder:14b'
print(f'Testing {MODEL} on GPU...')
start = time.time()

payload = json.dumps({
    'model': MODEL,
    'prompt': 'Write a Python one-liner to reverse a string. Answer in one line only.',
    'stream': False,
    'options': {'num_predict': 60}
}).encode()

req = urllib.request.Request(
    'http://localhost:11434/api/generate',
    data=payload,
    headers={'Content-Type': 'application/json'},
    method='POST'
)
with urllib.request.urlopen(req, timeout=120) as r:
    resp = json.load(r)

elapsed = time.time() - start
tokens = resp.get('eval_count', 0)
tok_per_sec = tokens / max(resp.get('eval_duration', 1) / 1e9, 0.001)

print(f'Response: {resp["response"].strip()}')
print(f'Speed: {tok_per_sec:.1f} tok/s  ({elapsed:.1f}s total)')
print(f'GPU check: {"✅ GPU" if tok_per_sec > 10 else "⚠️ CPU? Only " + str(round(tok_per_sec,1)) + " tok/s"}')

In [ ]:
# ── Cell 9: Run Evolution Gen 1 ───────────────────────────────────────
# Streams log output live — each LCB task should take ~3-8 min on GPU
import subprocess, sys, os

DEST = '/kaggle/working/SelfReplicatingAgent'
GEN_DIR = f'{DEST}/generations/gen_1'

env = os.environ.copy()
env['PYTHONPATH'] = GEN_DIR
env['OLLAMA_BASE_URL'] = 'http://localhost:11434'
env['OLLAMA_MODEL'] = 'qwen2.5-coder:14b'

proc = subprocess.Popen(
    [sys.executable, 'main.py', '--generation', '1'],
    cwd=GEN_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print('🚀 Evolution running — streaming output below:')
print('='*60)

# Stream output live
try:
    for line in proc.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    print('\n⚠️ Interrupted — killing process')
    proc.kill()

proc.wait()
print('='*60)
print(f'Evolution exited with code: {proc.returncode}')

In [ ]:
# ── Cell 10: View Results ─────────────────────────────────────────────
import json, os

DEST = '/kaggle/working/SelfReplicatingAgent'
progress_file = f'{DEST}/data/gen_1_progress.json'

if os.path.exists(progress_file):
    with open(progress_file) as f:
        data = json.load(f)

    results = data.get('results', {})
    task_list = data.get('task_list', [])

    success  = [tid for tid, r in results.items() if r.get('status') == 'success']
    failed   = [tid for tid, r in results.items() if r.get('status') == 'fail'
                and r.get('error_type') != 'SkippedUnsolvable']
    skipped  = [tid for tid, r in results.items() if r.get('error_type') == 'SkippedUnsolvable']

    print(f'Progress: {len(results)}/{len(task_list)} tasks')
    print(f'  ✅ Success:  {len(success)}')
    print(f'  ❌ Failed:   {len(failed)}')
    print(f'  ⏭️  Skipped:  {len(skipped)}')
    print()

    if success:
        print('Successful tasks:', success)
    if failed:
        print('\nFailed task details:')
        for tid in failed[:5]:
            r = results[tid]
            print(f"  {tid}: {r.get('error_type','')} — {r.get('stderr','')[:120]}")
else:
    print(f'No progress file yet at {progress_file}')

In [ ]:
# ── Cell 11: Save results to Kaggle output ────────────────────────────
import shutil, os

DEST = '/kaggle/working/SelfReplicatingAgent'
OUT  = '/kaggle/working/evolution_results'
os.makedirs(OUT, exist_ok=True)

# Copy data files (progress, logs)
for fname in ['gen_1_progress.json', 'evolution_log.json']:
    src = f'{DEST}/data/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{OUT}/{fname}')
        print(f'Saved: {fname}')

# Copy generation log
log = f'{DEST}/generations/gen_1/generation.log'
if os.path.exists(log):
    shutil.copy(log, f'{OUT}/generation_1.log')
    print('Saved: generation_1.log')

# Copy gen_2 if spawned
gen2 = f'{DEST}/generations/gen_2'
if os.path.exists(gen2):
    shutil.copytree(gen2, f'{OUT}/gen_2', dirs_exist_ok=True)
    print('Saved: gen_2 directory')

print(f'\nAll outputs in: {OUT}')
print('These will be available in Kaggle Output tab.')